In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss
from mlxtend.frequent_patterns import apriori,association_rules
import ipywidgets as widgets
from ipywidgets import interact

In [3]:
faceplate = pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Datasets\\Faceplate.csv")
faceplate

,Transaction,Red,White,Blue,Orange,Green,Yellow
0,1,1,1,0,0,1,0
1,2,0,1,0,1,0,0
2,3,0,1,1,0,0,0
3,4,1,1,0,1,0,0
4,5,1,0,1,0,0,0
5,6,0,1,1,0,0,0
6,7,1,0,1,0,0,0
7,8,1,1,1,0,1,0
8,9,1,1,1,0,0,0
9,10,0,0,0,0,0,1


In [4]:
faceplate=faceplate.astype(bool)
itemsets=apriori(faceplate,min_support=0.2,use_colnames=True)
rules=association_rules(itemsets,metric='confidence', min_threshold=0.6)
rules=rules[['antecedents','consequents','support','confidence','lift']]
relv_rules=rules[rules['lift']>1]
relv_rules.sort_values('lift',ascending=False)

C:\Users\PGCP-AI\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


,antecedents,consequents,support,confidence,lift
45,frozenset({Green}),"frozenset({White, Red, Transaction})",0.2,1.000000,2.500000
37,frozenset({Green}),"frozenset({Red, White})",0.2,1.000000,2.500000
42,"frozenset({Transaction, Green})","frozenset({Red, White})",0.2,1.000000,2.500000
11,frozenset({Green}),frozenset({Red}),0.2,1.000000,1.666667
23,"frozenset({Green, Transaction})",frozenset({Red}),0.2,1.000000,1.666667
35,"frozenset({Green, White})",frozenset({Red}),0.2,1.000000,1.666667
25,frozenset({Green}),"frozenset({Red, Transaction})",0.2,1.000000,1.666667
40,"frozenset({White, Transaction, Green})",frozenset({Red}),0.2,1.000000,1.666667
44,"frozenset({White, Green})","frozenset({Red, Transaction})",0.2,1.000000,1.666667
29,"frozenset({Orange, Transaction})",frozenset({White}),0.2,1.000000,1.428571


## Cosmetics

In [5]:
cosm=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Datasets\\Cosmetics.csv",index_col=0)
cosm

,Bag,Blush,Nail Polish,Brushes,Concealer,Eyebrow Pencils,Bronzer,Lip liner,Mascara,Eye shadow,Foundation,Lip Gloss,Lipstick,Eyeliner
Trans.,,,,,,,,,,,,,,
1,0,1,1,1,1,0,1,1,1,0,0,0,0,1
2,0,0,1,0,1,0,1,1,0,0,1,1,0,0
3,0,1,0,0,1,1,1,1,1,1,1,1,1,0
4,0,0,1,1,1,0,1,0,0,0,1,0,0,1
5,0,1,0,0,1,0,1,1,1,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,0,0,0,0,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,1,1,0,0
998,0,1,1,1,1,0,0,1,0,0,1,0,1,1


In [6]:
cosm=cosm.astype(bool)
itemsets=apriori(cosm,min_support=0.2,use_colnames=True)
rules=association_rules(itemsets,metric='confidence', min_threshold=0.6)
rules=rules[['antecedents','consequents','support','confidence','lift']]
relv_rules=rules[rules['lift']>1]
relv_rules.sort_values('lift',ascending=False)

,antecedents,consequents,support,confidence,lift
3,frozenset({Eye shadow}),frozenset({Mascara}),0.321,0.842520,2.359999
4,frozenset({Mascara}),frozenset({Eye shadow}),0.321,0.899160,2.359999
1,frozenset({Eyeliner}),frozenset({Concealer}),0.297,0.649891,1.470341
2,frozenset({Concealer}),frozenset({Eyeliner}),0.297,0.671946,1.470341
0,frozenset({Blush}),frozenset({Concealer}),0.220,0.606061,1.371178
6,frozenset({Foundation}),frozenset({Lip Gloss}),0.356,0.664179,1.355468
5,frozenset({Lip Gloss}),frozenset({Foundation}),0.356,0.726531,1.355468


In [7]:
# def gen_rules(permissible_support,confidence_threshold):
#     itemsets=apriori(cosm,min_support=permissible_support,use_colnames=True)
#     rules=association_rules(itemsets,metric='confidence', min_threshold=confidence_threshold)
#     rules=rules[['antecedents','consequents','support','confidence','lift']]
#     relv_rules=rules[rules['lift']>1]
#     relv_rules.sort_values('lift',ascending=False)

In [8]:
# interact(gen_rules,permissible_support=widgets.FloatSlider(value=0.3,min=0.01,max=1,step=0.01,description="Minimum support:"),
#         confidence_threshold=widgets.FloatSlider(value=0.3,min=0.01,max=1,step=0.01,description="Minimum support:"))

interactive(children=(FloatSlider(value=0.3, description='Minimum support:', max=1.0, min=0.01, step=0.01), Fl…

<function __main__.gen_rules(permissible_support, confidence_threshold)>

In [9]:
from ipywidgets import interact, widgets
from mlxtend.frequent_patterns import apriori, association_rules

def gen_rules(permissible_support, confidence_threshold):
    # 1. Generate itemsets
    itemsets = apriori(cosm, min_support=permissible_support, use_colnames=True)
    
    # 2. Check if itemsets is empty to avoid errors
    if itemsets.empty:
        return "No frequent itemsets found with this support level."
        
    # 3. Generate rules
    rules = association_rules(itemsets, metric='confidence', min_threshold=confidence_threshold)
    
    if rules.empty:
        return "No rules found with this confidence threshold."
        
    # 4. Filter and sort
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    relv_rules = rules[rules['lift'] > 1]
    
    # IMPORTANT: You must return the dataframe for it to show up in the widget
    return relv_rules.sort_values('lift', ascending=False)

In [12]:
interact(gen_rules, 
         permissible_support=widgets.FloatSlider(value=0.2, min=0.01, max=0.5, step=0.01, description="Support:"),
         confidence_threshold=widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description="Confidence:"))

interactive(children=(FloatSlider(value=0.2, description='Support:', max=0.5, min=0.01, step=0.01), FloatSlide…

<function __main__.gen_rules(permissible_support, confidence_threshold)>